# Internal Failure Modes: the Echo-Chamber Effect & Deterministic Circuit-Breakers

### Secure Agentic AI Systems (COMP9098), Week 5 in-class example

This standalone demo shows why a Planner/Executor/Validator pipeline is NOT automatically safe just because a "Validator" role exists in the diagram. If that Validator is itself a probabilistic model sharing the same framing as the Executor, it can simply agree with a wrong answer, an echo chamber, rather than actually verifying it. The fix is a deterministic Supervisor sitting above both: hard invariants that a model's "confidence" can never override.

Mock Executor + LLM Validator + deterministic Supervisor. No LLM and no API key are required anywhere in this file by default: the LLM Validator below is a scripted stand-in that reproduces, exactly, the over-confident approval a real model gives when asked to judge "does this look right" instead of "is this actually right". Set LIVE_LLM=1 (and have `openai` installed and your EDEN_TOKEN set, the same Eden AI key used throughout this course) to route the Validator through a real model instead; everything else in the demo is unchanged either way.

Points for the lecture:
- Scenario A (naive loop): Executor + naive LLM Validator only. The Validator scores semantic alignment, not ground truth, and approves a refund 100x the original charge. Nothing here raises an exception, the system "worked" and still did the wrong thing.
- Scenario B (hardened loop): the same Executor and Validator, but a deterministic Supervisor reviews every verdict against a hard invariant before anything commits, and a fingerprint trace detects when the loop is stuck in an echo chamber (the model re-justifying the same answer instead of correcting it) and escalates to a human.
- Deterministic invariants must always get the final word over probabilistic model verdicts (the External Policy Interceptors section's reference-monitor principle, applied one layer further in).

## Setup, run once

In [ ]:
from __future__ import annotations   # lets type hints reference classes defined later in this file

import hashlib                                 # used only to compute short, readable "fingerprints"
import os                                      # reads the LIVE_LLM env-var gate
from dataclasses import dataclass, field       # plain stdlib data containers, no external dependency
from enum import Enum                          # Verdict is a closed, whitelisted set of values
from typing import List, Optional              # type hints only, no runtime effect

## 1. The data the roles pass between each other

Plain stdlib, no Pydantic: this week's point is the CONTROL FLOW between roles, not schema validation (that was Week 4's lesson; the External Policy Interceptors section of this week covers it at the policy-engine layer instead).

In [ ]:
CCY = {"EUR", "USD", "GBP"}     # the currencies this toy system is willing to handle at all


@dataclass
class Task:
    """The ground truth the Executor is meant to act on. Known, fixed, trusted."""

    original_charge_eur: float    # what the customer actually paid, in EUR
    description: str = "Refund request"   # free-text label only, never read by any logic below


@dataclass
class RefundPayload:
    """The Executor's proposed action. Structurally valid is not the same as correct."""

    amount: float                          # the refund figure the Executor is proposing
    currency: str                          # must be one of CCY to pass type_check() below
    fingerprint: str = field(default="")   # stamped by the Executor, used for echo-chamber detection

    def type_check(self) -> bool:
        """Structural validity only: a positive amount in a currency we accept.
        Note what this does NOT check: whether the amount is the RIGHT amount."""
        # Two independent conditions, both must hold: a positive amount, AND a
        # currency this toy system recognises at all. Neither check looks at
        # the Task, so a wildly wrong amount still passes this just fine.
        return self.amount > 0 and self.currency in CCY


class Verdict(str, Enum):
    APPROVE = "APPROVE"     # the reviewing role is satisfied, the action may proceed
    REJECT = "REJECT"       # the reviewing role is NOT satisfied, the action must not proceed
    ESCALATE = "ESCALATE"   # not actually returned by any Review below; kept for completeness/future use


@dataclass
class Review:
    """A role's opinion on a payload. Confidence is NOT the same as correctness,
    that conflation is exactly the echo-chamber bug this demo exists to show."""

    verdict: Verdict      # one of the three values above
    confidence: float     # 0.0-1.0, how sure the reviewing role says it is, not how CORRECT it is
    reason: str           # short human-readable justification, printed in every trace line below


def fingerprint_of(payload: RefundPayload) -> str:
    """A short, stable hash of a proposal's shape. If the Executor proposes the
    exact same (wrong) answer twice in a row, the fingerprints match, that
    repetition is the signal a deterministic trace can catch even though
    neither probabilistic role ever flags anything itself."""
    # Hash only amount+currency (not the fingerprint field itself, which doesn't
    # exist yet at this point): two payloads with the same shape always get
    # the same fingerprint, regardless of when or how many times they're proposed.
    raw = f"{payload.amount:.2f}|{payload.currency}"
    return hashlib.sha1(raw.encode()).hexdigest()[:8]   # first 8 hex chars, plenty unique for a demo trace

## 2. The Executor

Proposes an action. Can never authorise its own output.

In [ ]:
class ExecutorAgent:
    """Confident, coherent, and in this demo, WRONG: it mistakes the task's
    currency-minor-unit cents figure for whole euros, a 100x unit confusion,
    the exact semantic-drift failure mode from this section's earlier slide."""

    name = "executor"   # used only for readability if you print role.name elsewhere; not read by the demo

    def propose(self, task: Task, _retry: int = 0) -> RefundPayload:
        # Deliberately reproduces a type-valid, semantically catastrophic
        # value: EUR 4,999.00 proposed against a EUR 49.99 charge. On a
        # retry after rejection, a purely reactive Executor just re-asserts
        # the same figure with more confidence, it does not re-derive it,
        # this is the "re-justifies instead of correcting" pattern.
        amount = round(task.original_charge_eur * 100, 2)   # the 100x unit-confusion bug, always the same
        payload = RefundPayload(amount=amount, currency="EUR")   # always EUR here, currency is never the issue
        payload.fingerprint = fingerprint_of(payload)             # stamp it so the Supervisor can trace repeats
        return payload   # handed to the Validator next; the Executor never sees anyone's verdict on it

## 3. The LLM Validator

Probabilistic. Scores semantic alignment, not truth. Mirrors the Internal Failure Modes section's diagram exactly: "sounds correct -> approve". It checks that the payload LOOKS like a plausible refund (right shape, right currency) and stops there. It never compares the amount back against the task's own ground truth, that comparison is precisely what the deterministic Supervisor below adds, and what this role is missing.

In [ ]:
class NaiveLLMValidator:
    name = "validator/llm"   # readability only, same as ExecutorAgent.name above

    def review(self, task: Task, payload: RefundPayload) -> Review:
        # The only branch point in this whole class: LIVE_LLM=1 switches from
        # the scripted mock to an actual model call. Unset (the default), it
        # always takes the mock path, so the lecture demo is fully deterministic.
        if os.environ.get("LIVE_LLM") == "1":
            return self._review_live(task, payload)   # optional real-model path, see markdown above
        return self._review_mock(payload)

    def _review_mock(self, payload: RefundPayload) -> Review:
        if not payload.type_check():
            # Only reachable if the Executor ever produced a malformed payload,
            # which it never does in this demo; kept as an honest safety branch.
            return Review(Verdict.REJECT, 0.99, "malformed payload")
        # Scripted to reproduce the lecture transcript precisely: it checks
        # shape and currency, sees both are fine, and stops there. Notice it
        # never looks at `task.original_charge_eur` at all, that's the bug.
        return Review(Verdict.APPROVE, 0.93, "currency matches, looks correct")

    def _review_live(self, task: Task, payload: RefundPayload) -> Review:   # pragma: no cover
        """Optional: route through a real model instead of the scripted mock.
        Requires `pip install openai` and EDEN_TOKEN set, the same Eden AI
        key used throughout this course (base_url is the only thing that
        makes this an Eden AI client rather than an OpenAI one). Not used
        unless LIVE_LLM=1 is set; the lecture demo never depends on this
        path."""
        from openai import OpenAI   # imported lazily so the file has zero hard dependencies by default

        # base_url is the only thing that makes this an Eden AI client rather than an OpenAI one.
        client = OpenAI(api_key=os.environ["EDEN_TOKEN"], base_url="https://api.edenai.run/v3")
        # The prompt deliberately only asks "does this look reasonable", the same
        # plausibility framing the mock above is scripted to reproduce, not
        # "recompute the correct refund and compare": that framing is the point.
        prompt = (
            f"A customer was charged EUR {task.original_charge_eur:.2f}. "
            f"An agent proposes refunding {payload.amount:.2f} {payload.currency}. "
            "Does this refund look reasonable? Answer APPROVE or REJECT and a one-line reason."
        )
        resp = client.chat.completions.create(
            model="qwen2.5-coder-32b-instruct",   # same model used throughout this course
            messages=[{"role": "user", "content": prompt}],   # single-turn call, no conversation history
        )
        text = resp.choices[0].message.content or ""              # the model's raw text reply
        verdict = Verdict.APPROVE if "APPROVE" in text.upper() else Verdict.REJECT   # crude keyword parse
        return Review(verdict, 0.5, text.strip()[:80])   # fixed placeholder confidence; truncated reason for the trace

## 4. The deterministic Supervisor

Hard invariants. Gets the final word. Plain Python, no model involved anywhere in this class. It holds the one invariant a probabilistic Validator can't be trusted to enforce on its own: the refund can never exceed the original charge.

In [ ]:
class DeterministicChecker:
    name = "supervisor"   # readability only, same as the other two roles' .name above

    def __init__(self) -> None:
        self.fingerprints: List[str] = []     # history, used by is_stuck() below

    def review(self, task: Task, payload: RefundPayload) -> Review:
        self.fingerprints.append(payload.fingerprint)   # record this attempt BEFORE deciding, so is_stuck() sees it
        if payload.amount > task.original_charge_eur:
            # The one hard invariant in this entire file: plain Python, no
            # model, no "confidence" score, just a direct comparison against
            # the Task's own ground truth, exactly what the Validator skipped.
            return Review(Verdict.REJECT, 1.0, "INVARIANT VIOLATION")
        return Review(Verdict.APPROVE, 1.0, "within original charge")

    def is_stuck(self) -> bool:
        """Echo-chamber / drift trace: true once the Executor has proposed
        the exact same (already-rejected) figure twice in a row, the signal
        that re-prompting alone will not fix this and a human must decide."""
        # Needs at least two recorded attempts to compare, and the two most
        # recent fingerprints must be identical: same amount, same currency,
        # i.e. the Executor did not change its answer at all on retry.
        return len(self.fingerprints) >= 2 and self.fingerprints[-1] == self.fingerprints[-2]

## 5. Scenario A: the naive loop

Executor + naive LLM Validator. No Supervisor.

In [ ]:
def run_scenario_a(task: Task) -> None:
    print(f"\n{'-' * 70}")
    print("# Scenario A: Executor + naive LLM Validator only")
    print("-" * 70)

    executor = ExecutorAgent()        # fresh instance, no state carried over from any previous run
    validator = NaiveLLMValidator()   # no Supervisor instantiated at all in this scenario, on purpose

    payload = executor.propose(task)   # one proposal, no retry loop here: Scenario A never asks for a second try
    print(f"executor       | Proposed refund: {payload.amount:.2f} {payload.currency}  [fp={payload.fingerprint}]")

    verdict = validator.review(task, payload)   # the ONLY check this scenario ever runs
    print(f"validator/llm  | Verdict={verdict.verdict.value}  conf={verdict.confidence:.2f}   -- {verdict.reason}")

    if verdict.verdict == Verdict.APPROVE:
        # Nothing else gets a vote: an APPROVE from the Validator commits
        # immediately, there is no second, independent check anywhere in this path.
        print(f"supervisor     | >>> COMMIT: refund {payload.amount:.2f} {payload.currency} approved")
        print(f"               # >> a EUR {payload.amount:,.2f} refund just cleared "
              f"a EUR {task.original_charge_eur:,.2f} charge")

    print("\nThe echo chamber held: semantic alignment, not verification. Nothing threw an exception.")

## 6. Scenario B: the hardened loop

Adds the deterministic Supervisor and a bounded retry budget with echo-chamber detection.

In [ ]:
def run_scenario_b(task: Task, max_attempts: int = 3) -> None:
    print(f"\n{'-' * 70}")
    print("# Scenario B: Executor + naive LLM Validator + deterministic Supervisor")
    print("-" * 70)

    executor = ExecutorAgent()            # same Executor class as Scenario A, still proposes the same bad figure
    validator = NaiveLLMValidator()       # same Validator class as Scenario A, still soft-approves every time
    supervisor = DeterministicChecker()   # the one thing Scenario A did not have

    for attempt in range(1, max_attempts + 1):   # bounded: this loop can run at most max_attempts times, never forever
        payload = executor.propose(task, _retry=attempt - 1)   # _retry is informational only, doesn't change the figure
        print(f"\nattempt {attempt}")
        print(f"executor       | Proposed refund: {payload.amount:.2f} {payload.currency}  [fp={payload.fingerprint}]")

        soft = validator.review(task, payload)   # the SAME naive check as Scenario A, still just a soft opinion here
        print(f"validator/llm  | Verdict={soft.verdict.value}  conf={soft.confidence:.2f}   -- {soft.reason}  (SOFT APPROVE)")

        hard = supervisor.review(task, payload)   # the deterministic check Scenario A never had; this is what actually decides
        if hard.verdict == Verdict.REJECT:
            print(f"supervisor     | Verdict={hard.verdict.value}  conf={hard.confidence:.2f}   -- {hard.reason}  (HARD REJECT)")

            if supervisor.is_stuck():
                # The Executor just proposed the identical (already-rejected)
                # figure again: no point letting it try a third time, escalate
                # to a human right now instead of burning the rest of the budget.
                print(f"supervisor     | >>> ESCALATE TO HUMAN: Executor re-proposed an identical, "
                      f"already-rejected figure [fp={payload.fingerprint}]")
                print(f"               # >> bounded retry budget + fingerprint match stopped a runaway loop "
                      f"after {attempt} attempt(s)")
                return   # exits the function entirely, the for-loop never reaches max_attempts here
            continue   # not stuck yet (this was the first rejection): let the loop try again, up to max_attempts

        # hard.verdict was APPROVE: the Supervisor's own invariant check passed,
        # independent of whatever the Validator said, so this is the only path
        # in the whole file where a refund actually commits for real.
        print(f"supervisor     | >>> COMMIT: refund {payload.amount:.2f} {payload.currency} approved  (HARD APPROVE)")
        return

    # Only reached if every attempt up to max_attempts was rejected WITHOUT ever
    # tripping is_stuck() (not possible with this particular Executor, which always
    # proposes the same figure, but kept as an honest fallback for a different one).
    print(f"supervisor     | >>> ESCALATE TO HUMAN: retry budget of {max_attempts} exhausted without a valid payload")

## 7. Demo

Run both scenarios and compare.

In [ ]:
print("COMP9098: Internal Failure Modes demo, echo-chamber effect & circuit-breakers")

task = Task(original_charge_eur=49.99, description="Customer refund request")   # the one shared ground truth

run_scenario_a(task)
# Expect: APPROVE then COMMIT. The naive loop has no independent check
# against the task's own ground truth, so a 100x refund clears cleanly.

run_scenario_b(task)
# Expect: SOFT APPROVE from the Validator every time (it never learns),
# but HARD REJECT from the Supervisor's invariant check. The Executor
# re-proposes the identical figure on retry, the fingerprint trace
# catches that it is stuck, and the loop escalates to a human instead
# of either committing the bad refund or retrying forever.

print(f"\n{'-' * 70}")
print("Takeaways:")
print("  - Deterministic invariants must always get the final word over probabilistic model verdicts.")
print("  - Bounded counters guarantee system termination, ensuring loops cannot run infinitely.")
print("  - Fingerprint and variance tracing actively detect when an agentic loop is stuck in an echo chamber.")
print("  - Hard software-engineering controls must always sit above soft probabilistic agreements.")
print("-" * 70)